# GPT 5.2 Dynamic Submission Evaluation

Evaluating submission against test labels.

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({"figure.dpi": 150, "axes.titlesize": 14, "axes.labelsize": 12})
sns.set_theme(style="whitegrid", palette="colorblind")

In [2]:
SUBMISSION_PATH = Path("/Users/hanneswidera/Uni/Master/thesis/GAIC_thesis/submissions/gpt5.2_dynamic/submission.jsonl")
LABELS_PATH = Path("/Users/hanneswidera/Uni/Master/thesis/GAIC_thesis/data/GAIC-2026/data/test_labels.jsonl")

print(f"Submission: {SUBMISSION_PATH}")
print(f"Labels: {LABELS_PATH}")
print(f"Submission exists: {SUBMISSION_PATH.exists()}")
print(f"Labels exist: {LABELS_PATH.exists()}")

Submission: /Users/hanneswidera/Uni/Master/thesis/GAIC_thesis/submissions/gpt5.2_dynamic/submission.jsonl
Labels: /Users/hanneswidera/Uni/Master/thesis/GAIC_thesis/data/GAIC-2026/data/test_labels.jsonl
Submission exists: True
Labels exist: True


In [3]:
def dataset_from_id(sample_id: str) -> str:
    """Extract dataset name from sample ID (e.g., 'ABSTRCT-test-1' -> 'ABSTRCT')."""
    return sample_id.rsplit("-", 2)[0]


def load_jsonl(path: Path) -> dict[str, str]:
    """Load JSONL file as {id: label} dict."""
    data = {}
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            data[record["id"]] = record["label"]
    return data

In [4]:
predictions = load_jsonl(SUBMISSION_PATH)
labels = load_jsonl(LABELS_PATH)

print(f"Predictions: {len(predictions)} samples")
print(f"Labels: {len(labels)} samples")

# Check overlap
common_ids = set(predictions.keys()) & set(labels.keys())
print(f"Common IDs: {len(common_ids)}")

if len(common_ids) < len(predictions):
    missing = set(predictions.keys()) - set(labels.keys())
    print(f"WARNING: {len(missing)} predictions have no labels")

Predictions: 4420 samples
Labels: 4420 samples
Common IDs: 4420


In [5]:
from dataclasses import dataclass
from typing import Any

@dataclass
class DatasetMetrics:
    """Metrics for a single dataset or overall."""
    n_samples: int
    macro_f1: float
    accuracy: float
    argument_f1: float
    no_argument_f1: float

    def format_row(self, name: str) -> str:
        return (
            f"{name:<12} {self.n_samples:>6} "
            f"{self.macro_f1:>10.4f} {self.accuracy:>8.4f} "
            f"{self.argument_f1:>8.4f} {self.no_argument_f1:>10.4f}"
        )


def compute_metrics(
    predictions: dict[str, str],
    labels: dict[str, str],
    dataset: str | None = None,
) -> DatasetMetrics | None:
    """Compute metrics for predictions against labels."""
    y_true, y_pred = [], []
    for sample_id, true_label in labels.items():
        if sample_id in predictions:
            if dataset is None or dataset_from_id(sample_id) == dataset:
                y_true.append(true_label)
                y_pred.append(predictions[sample_id])

    if not y_true:
        return None

    report: dict[str, Any] = classification_report(
        y_true, y_pred,
        target_names=["Argument", "No-Argument"],
        digits=4, output_dict=True, zero_division=0,
    )

    return DatasetMetrics(
        n_samples=len(y_true),
        macro_f1=report["macro avg"]["f1-score"],
        accuracy=report["accuracy"],
        argument_f1=report["Argument"]["f1-score"],
        no_argument_f1=report["No-Argument"]["f1-score"],
    )

In [6]:
def evaluate(predictions: dict[str, str], labels: dict[str, str]) -> dict:
    """Evaluate predictions against labels (overall and per-dataset)."""
    if not labels:
        print("ERROR: No labels found")
        return {}

    if not predictions:
        print("ERROR: No predictions found")
        return {}

    datasets = sorted(set(dataset_from_id(id_) for id_ in predictions.keys()))

    per_dataset = {}
    for dataset in datasets:
        metrics = compute_metrics(predictions, labels, dataset)
        if metrics:
            per_dataset[dataset] = metrics

    overall = compute_metrics(predictions, labels)

    return {"overall": overall, "per_dataset": per_dataset}


results = evaluate(predictions, labels)

In [ ]:
# Build complete classification report table (datasets as rows, metrics as columns)
def get_report_dict(predictions: dict[str, str], labels: dict[str, str], dataset: str | None = None) -> dict:
    """Get classification report as dict."""
    y_true, y_pred = [], []
    for sample_id, true_label in labels.items():
        if sample_id in predictions:
            if dataset is None or dataset_from_id(sample_id) == dataset:
                y_true.append(true_label)
                y_pred.append(predictions[sample_id])

    if not y_true:
        return {}

    return classification_report(y_true, y_pred, digits=4, output_dict=True, zero_division=0)


# Build rows for each dataset
rows = []
datasets = sorted(set(dataset_from_id(id_) for id_ in predictions.keys()))

for dataset in datasets + ["OVERALL"]:
    report = get_report_dict(predictions, labels, None if dataset == "OVERALL" else dataset)
    if report:
        rows.append({
            "Dataset": dataset,
            "N": int(report.get("macro avg", {}).get("support", 0)),
            "Accuracy": report.get("accuracy", 0),
            "Arg Precision": report.get("Argument", {}).get("precision", 0),
            "Arg Recall": report.get("Argument", {}).get("recall", 0),
            "Arg F1": report.get("Argument", {}).get("f1-score", 0),
            "NoArg Precision": report.get("No-Argument", {}).get("precision", 0),
            "NoArg Recall": report.get("No-Argument", {}).get("recall", 0),
            "NoArg F1": report.get("No-Argument", {}).get("f1-score", 0),
            "Macro F1": report.get("macro avg", {}).get("f1-score", 0),
            "Weighted F1": report.get("weighted avg", {}).get("f1-score", 0),
        })

df_report = pd.DataFrame(rows)

# Style the table
styled_report = df_report.style.background_gradient(
    subset=["Accuracy", "Arg Precision", "Arg Recall", "Arg F1", 
            "NoArg Precision", "NoArg Recall", "NoArg F1", "Macro F1", "Weighted F1"],
    cmap="RdYlGn", vmin=0.3, vmax=0.9
).format({
    "Accuracy": "{:.4f}",
    "Arg Precision": "{:.4f}",
    "Arg Recall": "{:.4f}",
    "Arg F1": "{:.4f}",
    "NoArg Precision": "{:.4f}",
    "NoArg Recall": "{:.4f}",
    "NoArg F1": "{:.4f}",
    "Macro F1": "{:.4f}",
    "Weighted F1": "{:.4f}",
})
display(styled_report)

In [ ]:
# Export table to markdown format (manual formatting, no tabulate needed)
def df_to_markdown(df: pd.DataFrame) -> str:
    """Convert DataFrame to markdown table."""
    cols = df.columns.tolist()
    header = "| " + " | ".join(cols) + " |"
    separator = "| " + " | ".join(["---"] * len(cols)) + " |"
    
    rows = []
    for _, row in df.iterrows():
        formatted = []
        for col in cols:
            val = row[col]
            if isinstance(val, float):
                formatted.append(f"{val:.4f}")
            else:
                formatted.append(str(val))
        rows.append("| " + " | ".join(formatted) + " |")
    
    return "\n".join([header, separator] + rows)

md_table = df_to_markdown(df_report)
print(md_table)

# Also save to file
md_path = SUBMISSION_PATH.parent / "results_table.md"
with open(md_path, "w") as f:
    f.write("# GPT 5.2 Dynamic - Test Set Results\n\n")
    f.write(md_table)
print(f"\n\nSaved to: {md_path}")

In [8]:
# Display results table
HEADER = f"{'Dataset':<12} {'N':>6} {'Macro-F1':>10} {'Acc':>8} {'Arg-F1':>8} {'NoArg-F1':>10}"
SEP = "-" * 60

print("=" * 60)
print("GPT 5.2 Dynamic - Test Set Evaluation")
print("=" * 60)
print()
print(HEADER)
print(SEP)

for dataset, metrics in sorted(results["per_dataset"].items()):
    print(metrics.format_row(dataset))

print(SEP)
if results["overall"]:
    print(results["overall"].format_row("TOTAL"))
print("=" * 60)

GPT 5.2 Dynamic - Test Set Evaluation

Dataset           N   Macro-F1      Acc   Arg-F1   NoArg-F1
------------------------------------------------------------
ABSTRCT         340     0.8735   0.8735   0.8724     0.8746
ACQUA           340     0.7882   0.7882   0.7882     0.7882
AEC             340     0.9141   0.9147   0.9068     0.9214
AFS             340     0.7119   0.7176   0.7526     0.6712
ARGUMINSCI      340     0.7933   0.7941   0.7799     0.8066
FINARG          340     0.4764   0.5235   0.3193     0.6335
IAM             340     0.7146   0.7147   0.7087     0.7205
PE              340     0.6646   0.6765   0.7277     0.6014
SCIARK          340     0.6682   0.6735   0.7102     0.6263
TACO            340     0.8408   0.8412   0.8333     0.8483
TAPE            340     0.7604   0.7676   0.7189     0.8020
TAUS            340     0.7853   0.7882   0.7600     0.8105
USELEC          340     0.7029   0.7059   0.7326     0.6732
------------------------------------------------------------

In [9]:
# Create DataFrame for visualization
rows = []
for dataset, m in results["per_dataset"].items():
    rows.append({
        "Dataset": dataset,
        "N": m.n_samples,
        "Macro-F1": m.macro_f1,
        "Accuracy": m.accuracy,
        "Argument F1": m.argument_f1,
        "No-Argument F1": m.no_argument_f1,
    })

df = pd.DataFrame(rows)
df = df.sort_values("Macro-F1", ascending=False)

# Style the dataframe
styled = df.style.background_gradient(
    subset=["Macro-F1", "Accuracy", "Argument F1", "No-Argument F1"],
    cmap="RdYlGn", vmin=0.3, vmax=0.9
).format({
    "Macro-F1": "{:.4f}",
    "Accuracy": "{:.4f}",
    "Argument F1": "{:.4f}",
    "No-Argument F1": "{:.4f}",
})
display(styled)

,Dataset,N,Macro-F1,Accuracy,Argument F1,No-Argument F1
2,AEC,340,0.9141,0.9147,0.9068,0.9214
0,ABSTRCT,340,0.8735,0.8735,0.8724,0.8746
9,TACO,340,0.8408,0.8412,0.8333,0.8483
4,ARGUMINSCI,340,0.7933,0.7941,0.7799,0.8066
1,ACQUA,340,0.7882,0.7882,0.7882,0.7882
11,TAUS,340,0.7853,0.7882,0.7600,0.8105
10,TAPE,340,0.7604,0.7676,0.7189,0.8020
6,IAM,340,0.7146,0.7147,0.7087,0.7205
3,AFS,340,0.7119,0.7176,0.7526,0.6712
12,USELEC,340,0.7029,0.7059,0.7326,0.6732


In [13]:
# Save metrics to file
metrics_path = SUBMISSION_PATH.parent / "test_metrics.txt"

with open(metrics_path, "w") as f:
    f.write("GPT 5.2 Dynamic - Test Set Evaluation\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Submission: {SUBMISSION_PATH}\n")
    f.write(f"Labels: {LABELS_PATH}\n\n")
    f.write(HEADER + "\n")
    f.write(SEP + "\n")
    for dataset, metrics in sorted(results["per_dataset"].items()):
        f.write(metrics.format_row(dataset) + "\n")
    f.write(SEP + "\n")
    if results["overall"]:
        f.write(results["overall"].format_row("TOTAL") + "\n")

print(f"Metrics saved to: {metrics_path}")

Metrics saved to: /Users/hanneswidera/Uni/Master/thesis/GAIC_thesis/submissions/gpt5.2_dynamic/test_metrics.txt
